# Lesson 20 · GANs — adversarial networks: two networks compete, one forging digits, the other catching the fakes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/20-gan-en.ipynb)

This notebook accompanies [Lesson 20 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gan/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/20-gan.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We write a GAN ourselves in numpy: a generator G turns noise into samples, a discriminator D guesses whether a
sample is real or fake, and the two networks learn by competing against each other. The central question: **what
does this game actually minimize, and how do we know it has won?**

**What you'll do:**
1. Write the generator's two losses and see why the original loss $\log(1 - D(G(z)))$ saturates.
2. Compute the exact best discriminator $D^*$ in a one-dimensional game, check two identities involving the JSD, and
   see where each loss takes the generator.
3. Train a two-dimensional GAN on a ring of 8 clusters, with and without the R1 gradient penalty, and see mode
   collapse.
4. Generate MNIST digits with the demo's generator and measure the distance from each sample to the nearest training
   image.
5. Evaluate: why $D \approx \tfrac12$ doesn't mean success (FD, precision/recall, a fresh classifier).

**Background:** [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/)
(binary cross-entropy), [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/),
the Adam optimizer of [Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/), and
[Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/) (the decoder,
the FD metric). **Run time:** under a minute on a CPU, including the first download of MNIST (~11 MB) and of the
demo's 1.1 MB of results.

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. Only the full
training runs on MNIST (15 configurations × 3 seeds, 20,000 steps each) are loaded from the very files the demo
uses; the notebook reruns a small version of each mechanism and checks whatever can be recomputed.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label):
# real data = blue, circles; generator samples = orange, squares; original loss = green, solid line
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def span(values, digits=2):
    """The min–max range over the seeds, as in the demo: 7.88–8.02."""
    lo, hi = min(values), max(values)
    return num(lo, digits) if round(lo, digits) == round(hi, digits) else f"{num(lo, digits)}–{num(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 6.2·10⁻⁵."""
    e = math.floor(math.log10(abs(x)))
    return f"{num(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it is
    long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def tile(images, rows, cols, gap=2):
    """Tile rows × cols 28×28 images into one large image (much faster than one axis per image); the gaps are 0."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))  # no overflow for large |a|


def softplus(a):
    return np.maximum(a, 0) + np.log1p(np.exp(-np.abs(a)))  # log(1 + eᵃ), numerically stable


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.887), span([7.88, 8.02]), sci(6.2e-5))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits. The
cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the same
data. As in the demo's runs (and Lesson 19), the GAN learns from the first 55,000 training images; the 10,000 test
images are used only for evaluation.

The cell also loads the four results files that the demo uses (`fetch_json`, with an MD5 check):
- `gan-1d.json`: the one-dimensional game with the exact $D^*$ (section 4);
- `gan-2d.json`: the two-dimensional GAN on the ring of 8 clusters, 5 seeds × 20,000 steps for each configuration
  (section 5);
- `gan-mnist.json`: every GAN run on MNIST, 3 seeds per configuration, along with their measurements (sections 7–8);
- `gan-g16.json`: the best generator (seed 0), in the same format as Lesson 19's decoder (section 6).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of the long training runs (the same files as the demo on the web). build.py replaces COURSE_DATA_URL
# with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
one = fetch_json("gan-1d.json", "db1315153556b0b70774b9a9a4685439")
two = fetch_json("gan-2d.json", "b8b7c9bd3dee0ab24d66dcd40e5282a8")
mn = fetch_json("gan-mnist.json", "79bcc429034a8648347eb6999d9d27cd")
g16 = fetch_json("gan-g16.json", "fbb94a427bde0d1a5f3e042eeaf49c26")
size = sum((Path("data") / f).stat().st_size for f in ["gan-1d.json", "gan-2d.json", "gan-mnist.json", "gan-g16.json"])
print("training:", train_img.shape, " test:", test_img.shape, f" · the demo's 4 files: {num(size / 1e6, 1)} MB")

In [ ]:
Xtr = train_img[:55000].astype(np.float32) / 255  # the 55,000 images the GAN learns from, grayscale pixels in [0, 1]
Xte = test_img.astype(np.float32) / 255  # 10,000 test images
print(f"training {num(len(Xtr), 0)} · test {num(len(Xte), 0)} images · mean brightness {pct(Xtr.mean())}")

## 3 · Two networks, two losses

A GAN has two networks. The **generator** $G$ turns a noise vector $z \sim \mathcal N(0, I)$ into a sample $G(z)$
(here $z$ is the input noise, like the latent code of Lesson 19, not the weighted sum $z = xW + b$ of Lesson 01).
The **discriminator** $D$ takes a sample $x$ and returns a logit $a(x)$, with $D(x) = \sigma(a(x))$ the probability
it gives to $x$ being real. $D$ learns like the binary classifier of Lesson 04, with label 1 for real samples and 0
for $G$'s samples:

$$\mathcal L_D = \mathbb E_x\big[-\log D(x)\big] + \mathbb E_z\big[-\log(1 - D(G(z)))\big]
= \mathbb E_x\big[\mathrm{softplus}(-a(x))\big] + \mathbb E_z\big[\mathrm{softplus}(a(G(z)))\big]$$

with $\mathrm{softplus}(u) = \log(1 + e^u)$. The generator has two options:
- the **original loss** of the minimax game (Goodfellow et al., 2014): minimize $\log(1 - D(G(z))) =
  -\mathrm{softplus}(a)$;
- the **non-saturating loss**, which almost everyone uses: minimize $-\log D(G(z)) = \mathrm{softplus}(-a)$.

With respect to the logit $a$ of a fake sample, the gradients of the two losses are $-\sigma(a) = -D$ and
$-(1 - \sigma(a)) = -(1 - D)$: the same direction (increase $a$, that is, make $D$ believe the sample is real), but
different magnitudes. The cell below checks these two derivatives with finite differences, then plots their
magnitudes against $D(G(z))$. The dotted line marks the actual value of $D(G(z))$ at step 500 of an MNIST run with
the original loss (seed 0, read from `gan-mnist.json`).

In [ ]:
D_fake = np.logspace(-6, 0, 401)[:-1]  # D(G(z)) from 10⁻⁶ to nearly 1
a = np.log(D_fake) - np.log1p(-D_fake)  # logit, D = σ(a)
g_sat, g_ns = -sigmoid(a), -sigmoid(-a)  # ∂/∂a of −softplus(a) and of softplus(−a)
h = 1e-6
assert np.allclose((-softplus(a + h) + softplus(a - h)) / (2 * h), g_sat, rtol=1e-6, atol=1e-12)
assert np.allclose((softplus(-a - h) - softplus(-a + h)) / (2 * h), g_ns, rtol=1e-6, atol=1e-12)

sat_traj = mn["runs"]["sat-r0-lr0.001"]["seeds"][0]["traj"]
p500 = next(t for t in sat_traj if t["step"] == 500)
d500 = p500["DGz"]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.loglog(D_fake, -g_ns, color=ORANGE, lw=2.5, ls="--", label="non-saturating −log D: |∂/∂a| = 1 − D")
ax.loglog(D_fake, -g_sat, color=GREEN, lw=2.5, label="original loss log(1 − D): |∂/∂a| = D")
ax.axvline(d500, color=GREY, lw=1.2, ls=":")
ax.text(d500 * 1.4, 1e-2, "MNIST, original loss,\nstep 500", fontsize=8, color="black")
ax.set(
    xlabel="D(G(z)): the probability D gives to a fake being real",
    ylabel="gradient magnitude w.r.t. logit a",
    ylim=(5e-7, 3),
)
ax.set_title("When D rejects easily, the original loss is almost flat")
ax.legend(frameon=False, fontsize=9, loc="lower right")
fmt_axes(fig)
plt.show()
print(f"step 500 (MNIST, original loss, seed 0): mean D(G(z)) = {sci(d500)}")
print(f"  one sample: |∂/∂a| original {sci(d500)}, non-saturating {num(1 - d500, 5)} → {num((1 - d500) / d500, 0)} times larger")  # fmt: skip
print(f"  the whole network G (gradient norm over 256 fixed noise vectors): non-saturating {num(p500['gNS'], 2)}, original {sci(p500['gSAT'])}"
      f" → {num(p500['gNS'] / p500['gSAT'], 0)} times larger (≈ {num(round(p500['gNS'] / p500['gSAT'], -3), 0)})")  # fmt: skip

The two curves meet at $D = \tfrac12$. On the left, when $D$ is sure the sample is fake ($D(G(z)) \to 0$), the
gradient of the original loss is proportional to $D$ itself and dies out, while that of the non-saturating loss
stays close to 1. That is exactly the situation early in training: $G$ is still bad, and $D$ easily spots the
fakes. Goodfellow et al. write that $\log(1 - D(G(z)))$ then *saturates*, and suggest having $G$ maximize
$\log D(G(z))$ instead: the same fixed point, but much stronger gradients early on.

In the MNIST run above, $D(G(z))$ is already down to $6.2 \cdot 10^{-5}$ at step 500: the original loss gives $G$ a
gradient about 14,000 times smaller, and $G$ never escapes (section 7).

## 4 · A one-dimensional game with the best discriminator

To see what the game actually minimizes, we take an example small enough to compute everything exactly, like the
demo's first card. The data are two Gaussian clusters,
$p_{\text{data}} = \tfrac12\mathcal N(-2, 0.5^2) + \tfrac12\mathcal N(2, 0.5^2)$. The generator has only two
parameters: $G(z) = \mu + s\,z$ with $z \sim \mathcal N(0, 1)$, so $p_g = \mathcal N(\mu, s^2)$.

For a fixed $G$, $D$ maximizes $\mathbb E_x \log D(x) + \mathbb E_z \log(1 - D(G(z)))$ at each point $x$
independently, and the solution is (Goodfellow et al., 2014, Proposition 1):

$$D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$$

The cell below writes $p_{\text{data}}$, $p_g$ and $D^*$ in the log domain, to avoid underflow in the tails of the
distributions.

In [ ]:
MIX = ((-2.0, 0.5, 0.5), (2.0, 0.5, 0.5))  # (center, standard deviation, weight) of the two clusters
assert [list(c) for c in MIX] == one["mix"]  # exactly the demo's data
LOG2PI = math.log(2 * math.pi)


def log_comps(x):
    """log(weight × density) of each cluster of p_data; an array of 2 × (the size of x)."""
    return np.stack([np.log(w) - 0.5 * ((x - m) / s) ** 2 - np.log(s) - 0.5 * LOG2PI for m, s, w in MIX])


def log_pdata(x):
    return np.logaddexp(*log_comps(x))


def log_pg(x, mu, s):
    return -0.5 * ((x - mu) / s) ** 2 - np.log(s) - 0.5 * LOG2PI


def d_star(x, mu, s):
    """D*(x) = p_data / (p_data + p_g)."""
    a = log_pdata(x)
    return np.exp(a - np.logaddexp(a, log_pg(x, mu, s)))


print(f"generator N(0.3, 0.5²): D*(−2) = {num(d_star(-2.0, 0.3, 0.5), 4)} · D*(0.3) = {num(d_star(0.3, 0.3, 0.5), 4)}"
      f" · D*(2) = {num(d_star(2.0, 0.3, 0.5), 4)}")  # fmt: skip

### 4.1 · A real network learns nearly $D^*$

$D^*$ is the solution over *all functions*. Does a small network that learns from samples get close to it? The cell
below writes a general ReLU MLP (backpropagation by hand, as in Lesson 06; section 5 reuses it for both $G$ and
$D$), then trains $D$ = 1 → 32 → 32 → 1 for 2,000 steps, each step on 256 real samples and 256 samples from the
fixed generator $\mathcal N(0.3,\, 0.5^2)$.

The `MLP` class holds R independent networks with the same architecture at once: each weight matrix gets an extra
leading dimension R. Here R = 1; section 5 trains 12 GANs in parallel that way.

In [ ]:
class MLP:
    """R independent ReLU networks with the same architecture (sizes), run in parallel. All the parameters live in one
    flat array self.flat (W₀, b₀, W₁, b₁, … are views into it), so Adam is just a few operations on one array."""

    def __init__(self, sizes, R, rng):
        shapes = [s for a, b in itertools.pairwise(sizes) for s in ((R, a, b), (R, 1, b))]
        self.flat = np.zeros(sum(math.prod(s) for s in shapes))
        self.p, i = [], 0
        for s in shapes:
            self.p.append(self.flat[i : i + math.prod(s)].reshape(s))
            i += math.prod(s)
        for l, (a, b) in enumerate(itertools.pairwise(sizes)):  # He: factor 2 for hidden layers, 1 for output; b = 0
            self.p[2 * l][...] = rng.standard_normal((R, a, b)) * np.sqrt((1 if l == len(sizes) - 2 else 2) / a)
        self.m, self.v, self.t = np.zeros_like(self.flat), np.zeros_like(self.flat), 0

    def fwd(self, x):
        """x: R × B × number of inputs. Returns the input of each layer and, last, the output (the last layer is
        linear)."""
        hs = [x]
        for l in range(0, len(self.p), 2):
            u = hs[-1] @ self.p[l] + self.p[l + 1]  # row convention: h = f(xW + b)
            hs.append(np.maximum(u, 0) if l < len(self.p) - 2 else u)
        return hs

    def back(self, hs, d, params=True):
        """Backpropagate d = ∂L/∂(output): the flat gradient with respect to every parameter (if params) and
        ∂L/∂(input)."""
        g = []
        for l in range(len(self.p) - 2, -1, -2):
            if params:
                g += [d.sum(1, keepdims=True), hs[l // 2].transpose(0, 2, 1) @ d]  # ∂L/∂b, ∂L/∂W
            d = d @ self.p[l].transpose(0, 2, 1)
            if l > 0:
                d = d * (hs[l // 2] > 0)  # ReLU mask of the layer below
        return (np.concatenate([q.ravel() for q in g[::-1]]) if params else None), d

    def adam(self, g, lr, b1=0.5, b2=0.999):
        """Adam (Lesson 07). β₁ = 0.5, as in DCGAN and every GAN run of the demo."""
        self.t += 1
        self.m += (1 - b1) * (g - self.m)
        self.v += (1 - b2) * (g * g - self.v)
        self.flat -= lr * (self.m / (1 - b1**self.t)) / (np.sqrt(self.v / (1 - b2**self.t)) + 1e-8)

In [ ]:
MU0, S0 = 0.3, 0.5  # fixed generator G(z) = 0.3 + 0.5·z
rng = np.random.default_rng(0)
D1 = MLP([1, 32, 32, 1], 1, rng)
B = 256
for step in range(2000):
    xr = np.where(rng.random(B) < 0.5, -2.0, 2.0) + 0.5 * rng.standard_normal(B)  # real samples
    xf = MU0 + S0 * rng.standard_normal(B)  # samples from G
    hs = D1.fwd(np.r_[xr, xf][None, :, None])
    a = hs[-1][0, :, 0]
    da = np.r_[-sigmoid(-a[:B]), sigmoid(a[B:])] / B  # ∂L_D/∂a of the cross-entropy: label 1 real, 0 fake
    D1.adam(D1.back(hs, da[None, :, None])[0], 3e-3, b1=0.9)  # D learns alone: Adam's defaults

xs = np.linspace(-4, 4, 801)
D_learn = sigmoid(D1.fwd(xs[None, :, None])[-1][0, :, 0])
D_opt = d_star(xs, MU0, S0)
wt = np.exp(log_pdata(xs)) + np.exp(log_pg(xs, MU0, S0))
wt /= wt.sum()  # weights: where there are real or fake samples
err = (wt * np.abs(D_learn - D_opt)).sum()
half = xs[:-1][np.diff(np.sign(D_opt - 0.5)) != 0]
print(f"mean |learned D − D*| (where there are samples): {num(err, 4)}")
print("D* = ½ only at x ≈", " and ".join(num(v, 2) for v in half), "(where p_data = p_g)")
assert err < 0.03 and len(half) == 2

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.fill_between(xs, np.exp(log_pdata(xs)), color=BLUE, alpha=0.35, label="p_data (the data)")
ax.plot(xs, np.exp(log_pg(xs, MU0, S0)), color=ORANGE, lw=2.2, label="p_g = N(0.3, 0.5²)")
ax2 = ax.twinx()
ax2.plot(xs, D_opt, color="black", lw=1.6, ls="--", label="D*(x), right axis")
ax2.plot(xs, D_learn, color=GREEN, lw=2, ls=":", label="D learned from samples, right axis")
ax2.axhline(0.5, color=GREY, lw=0.8)
ax2.set(ylim=(0, 1.02), ylabel="probability of being real")
ax2.spines["right"].set_visible(True)
ax.set(xlabel="x", ylabel="density", ylim=(0, 0.85), xlim=(-4, 4))
ax.set_title("The best D for a fixed generator, and the D that an MLP learns")
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4)
fmt_axes(fig)
plt.show()

The blue area is the data, the orange curve the generator. The dashed black curve is $D^*$: close to 1 on the two
clusters (only data there), close to 0 in the middle (only fake samples there). The dotted green curve is the $D$
that the MLP learned from samples; it tracks $D^*$ closely wherever there are samples. Note that $D^* = \tfrac12$
at only two points, where the two densities cross: "$D = \tfrac12$" is a sign of success only when it holds
**everywhere**, that is, when $p_g = p_{\text{data}}$. Section 8 shows why looking at $D$ averaged over the samples
isn't enough.

### 4.2 · The game minimizes the JSD

Substitute $D^*$ into the value of the game $V(D, G) = \mathbb E_x \log D(x) + \mathbb E_z \log(1 - D(G(z)))$
(Goodfellow et al., 2014, Theorem 1):

$$V(D^*, G) = -\log 4 + 2\cdot \mathrm{JSD}(p_{\text{data}} \,\|\, p_g), \qquad
\mathrm{JSD}(p \,\|\, q) = \tfrac12 \mathrm{KL}(p \,\|\, m) + \tfrac12 \mathrm{KL}(q \,\|\, m),\;\; m = \tfrac{p + q}{2}$$

The JSD lies between 0 (the two distributions coincide) and $\ln 2$ (they don't overlap at all). The cell below
computes the JSD, the KL in the direction $\mathrm{KL}(p_g \,\|\, p_{\text{data}})$ (section 4.3 needs it) and
$V(D^*, G)$ by numerical integration on a fine grid, for a few generators.

In [ ]:
X = np.linspace(-16, 16, 64001)  # integration grid, as in gan-1d.json
DX = X[1] - X[0]
A_X = log_pdata(X)


def divergences(mu, s):
    """JSD(p_data ‖ p_g), KL(p_g ‖ p_data) (in nats) and V(D*, G), as Riemann sums on the grid X."""
    b = log_pg(X, mu, s)
    l = np.logaddexp(A_X, b)  # log(p_data + p_g)
    pa, pb = np.exp(A_X), np.exp(b)
    jsd = 0.5 * (pa * (A_X - l + np.log(2)) + pb * (b - l + np.log(2))).sum() * DX
    kl = (pb * (b - A_X)).sum() * DX
    v = ((pa * (A_X - l)).sum() + (pb * (b - l)).sum()) * DX  # E_data log D* + E_g log(1 − D*)
    return jsd, kl, v


print(f"{'generator':>16} {'JSD':>8} {'V(D*, G)':>10} {'−log 4 + 2·JSD':>16}")
for mu, s in [(0.3, 0.5), (2.0, 0.5), (0.0, 2.04), (6.0, 0.5)]:
    j, _, v = divergences(mu, s)
    print(f"  N({num(mu, 1)}, {num(s, 2)}²) {num(j, 4):>8} {num(v, 4):>10} {num(-math.log(4) + 2 * j, 4):>16}")
    assert abs(v - (-math.log(4) + 2 * j)) < 1e-6
print(f"ln 2 = {num(math.log(2), 4)}: the largest JSD, when the two distributions don't overlap")

In every row, $V(D^*, G)$ computed directly equals $-\log 4 + 2\cdot\mathrm{JSD}$ exactly. The generator
$\mathcal N(6,\, 0.5^2)$ barely overlaps the data, so its JSD nearly reaches $\ln 2$.

### 4.3 · Where does each loss take the generator?

In a training step, $G$ is updated with $D$ **held fixed**. Suppose $D = D^*$ of the current generator. The
gradients of the two losses with respect to $(\mu, s)$ go through $x = \mu + s\,z$ by the chain rule:
$\partial/\partial\mu = \partial/\partial x$ and $\partial/\partial s = z\,\partial/\partial x$. The expectation over
$z \sim \mathcal N(0, 1)$ is taken with the trapezoidal rule on $[-8, 8]$ with a step of 0.05, as in the demo. The
two identities to check:

- original loss: $\nabla\, \mathbb E_z \log(1 - D^*(G(z))) = \nabla\, 2\cdot\mathrm{JSD}$ (Goodfellow et al., 2014);
- non-saturating: $\nabla\, \mathbb E_z[-\log D^*(G(z))] = \nabla\,[\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot
  \mathrm{JSD}]$ (Arjovsky & Bottou, 2017, Theorem 2.5).

The cell below writes the analytic gradients of both losses, then compares them with finite differences of the JSD
and the KL at four points $(\mu, s)$, the very four points that the demo's `reel/gan/oned.py` checks.

In [ ]:
Z = np.arange(-8, 8 + 1e-9, 0.05)  # integration nodes for z ~ N(0, 1)
WZ = np.exp(-0.5 * Z**2)
WZ /= WZ.sum()


def gen_grads(mu, s):
    """Gradients with respect to (μ, s) of G's two losses, with D* held fixed at the current generator; takes arrays
    of (μ, s)."""
    mu, s = np.asarray(mu, float)[..., None], np.asarray(s, float)[..., None]
    x = mu + s * Z
    c = log_comps(x)
    a, b = np.logaddexp(*c), log_pg(x, mu, s)
    D = np.exp(a - np.logaddexp(a, b))  # D*(G(z))
    dA = (np.exp(c - a) * np.stack([-(x - m) / sd**2 for m, sd, _ in MIX])).sum(0)  # ∂/∂x log p_data
    dB = -(x - mu) / s**2  # ∂/∂x log p_g, with the (μ, s) of D* held fixed
    dl = D * dA + (1 - D) * dB  # ∂/∂x log(p_data + p_g)
    out = {}
    for k, dx in (("sat", dB - dl), ("ns", -(dA - dl))):  # ∂/∂x of log(1 − D*) and of −log D*
        out[k] = np.stack([(WZ * dx).sum(-1), (WZ * dx * Z).sum(-1)], -1)
    out["eD"] = (WZ * D).sum(-1)  # mean D*(G(z))
    return out


worst = 0.0
for mu, s in [(1.0, 0.5), (0.342, 1.674), (3.0, 1.0), (-1.2, 0.8)]:
    g, h = gen_grads(mu, s), 1e-5
    dj, dk = [], []
    for dm, ds in ((h, 0), (0, h)):
        jp, kp, _ = divergences(mu + dm, s + ds)
        jm, km, _ = divergences(mu - dm, s - ds)
        dj.append((jp - jm) / (2 * h))
        dk.append((kp - km) / (2 * h))
    dj, dk = np.array(dj), np.array(dk)
    worst = max(worst, np.abs(g["sat"] - 2 * dj).max(), np.abs(g["ns"] - (dk - 2 * dj)).max())
    print(f"(μ, s) = ({num(mu, 3)}, {num(s, 3)}): ∂/∂μ original {num(g['sat'][0], 5)} = 2·∂JSD {num(2 * dj[0], 5)} · "
          f"non-saturating {num(g['ns'][0], 5)} = ∂(KL − 2·JSD) {num(dk[0] - 2 * dj[0], 5)}")  # fmt: skip
print(f"largest gap between the analytic gradients and the finite differences: {sci(worst)}")
assert worst < 1e-5, "the two identities don't match: check gen_grads again"

Both identities hold up to the error of the finite differences. So with the optimal $D$, the original loss is
gradient descent on the JSD, just as the theory promises. The non-saturating loss does **not** minimize the JSD: it
follows $\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$. The KL in this direction (the "reverse" KL)
heavily penalizes putting probability where there is no data, but barely penalizes missing part of the data
(Arjovsky & Bottou, 2017).

The cell below shows the price of the original loss far from the data: for the generator $\mathcal N(6,\, 0.5^2)$,
$D^*$ is almost certain that every sample is fake, and the original loss's gradient with respect to $\mu$ is tens of
thousands of times smaller.

In [ ]:
g_far = gen_grads(6.0, 0.5)
print(f"N(6, 0.5²): mean D*(G(z)) = {sci(g_far['eD'])} · ∂/∂μ: original loss {sci(g_far['sat'][0])}, "
      f"non-saturating {num(g_far['ns'][0], 2)} → {num(g_far['ns'][0] / g_far['sat'][0], 0)} times larger")  # fmt: skip
grid_far = next(r for r in one["grid"] if r["mu"] == 6 and r["s"] == 0.5)
assert abs(g_far["sat"][0] / grid_far["gSat"] - 1) < 1e-3 and abs(g_far["ns"][0] / grid_far["gNs"] - 1) < 1e-3

### 🤔 Predict first
Starting from the generator $\mathcal N(0.3,\, 0.5^2)$ (the figure in section 4.1), run gradient descent on
$(\mu, s)$ with each loss, recomputing $D^*$ at every step. Do the two losses take the generator to **the same
place**? If not, which one covers both clusters?

In [ ]:
STARTS = [(0.3, 0.5), (0.3, 1.5), (-1.0, 0.3), (1.0, 2.0), (4.0, 0.5), (5.0, 1.0)]  # as in gan-1d.json


def flows(starts, steps=4000, lr=0.05, every=40):
    """The gradient flows of both losses from each starting point, all at once (D* recomputed at every step,
    s ≥ 0.05)."""
    st = np.array(starts, float)
    mu, s = np.r_[st[:, 0], st[:, 0]], np.r_[st[:, 1], st[:, 1]]
    sat = np.arange(len(mu)) < len(st)  # first half: the original loss; second half: non-saturating
    path = [np.c_[mu, s]]
    for t in range(1, steps + 1):
        g = gen_grads(mu, s)
        gr = np.where(sat[:, None], g["sat"], g["ns"])
        mu, s = mu - lr * gr[:, 0], np.maximum(0.05, s - lr * gr[:, 1])
        if t % every == 0:
            path.append(np.c_[mu, s])
    path = np.stack(path, 1)  # (2·number of points) × number of checkpoints × 2
    return path[: len(st)], path[len(st) :]


path_sat, path_ns = flows(STARTS)
print(f"{'start':>14} │ {'original loss → (μ, s), JSD':>28} │ {'non-saturating → (μ, s), JSD':>28}")
for i, (m0, s0) in enumerate(STARTS):
    row = []
    for path, key in ((path_sat, "sat"), (path_ns, "ns")):
        mu, s = path[i, -1]
        row.append(f"({num(mu, 2)}, {num(s, 2)}), {num(divergences(mu, s)[0], 4)}")
        assert np.allclose([mu, s], one["flows"][i][key]["end"], atol=2e-3), "differs from the demo"
    print(f"  ({num(m0, 1)}, {num(s0, 1)}) │ {row[0]:>28} │ {row[1]:>28}")

No. From $\mathcal N(0.3,\, 0.5^2)$, the **original loss** spreads the generator into a wide Gaussian
$\mathcal N(0,\, 2.04^2)$ that covers both clusters: exactly the single Gaussian with the smallest JSD. The
**non-saturating loss** commits to one cluster, $\mathcal N(2,\, 0.5^2)$. By the JSD, the wide Gaussian is better
(0.2010 versus 0.2157). But it puts a lot of probability in the empty gap between the two clusters, where there is
no data, and that is exactly what the reverse KL penalizes heavily. The cell below prints the two quantities at these
two endpoints, and at the third endpoint of the next paragraph.

From $(0.3,\, 1.5)$, the non-saturating loss also stops at a wide Gaussian, $\mathcal N(0,\, 1.71^2)$: that is a
local minimum of $\mathrm{KL} - 2\cdot\mathrm{JSD}$, not its best point.

In [ ]:
for name, (mu, s) in [("wide Gaussian of the original loss", path_sat[0, -1]), ("one cluster", path_ns[0, -1]),
                      ("wide Gaussian of non-saturating", path_ns[1, -1])]:  # fmt: skip
    j, k, _ = divergences(mu, s)
    print(f"{name:>34} N({num(mu, 2)}, {num(s, 2)}²): JSD {num(j, 4)} · KL(p_g ‖ p_data) {num(k, 3)} · "
          f"KL − 2·JSD {num(k - 2 * j, 3)}")  # fmt: skip


def jsd_grid(mus, ss, n=1601):
    """The JSD on a grid of (μ, s), with a coarser integration grid (only for drawing the contours)."""
    x = np.linspace(-16, 16, n)
    a = log_pdata(x)
    b = log_pg(x, mus[..., None], ss[..., None])
    l = np.logaddexp(a, b) - np.log(2)
    return 0.5 * (np.exp(a) * (a - l) + np.exp(b) * (b - l)).sum(-1) * (x[1] - x[0])


M_, S_ = np.meshgrid(np.linspace(-4, 6, 81), np.linspace(0.2, 3, 57))
fig, ax = plt.subplots(figsize=(8, 4.4))
cs = ax.contour(M_, S_, jsd_grid(M_, S_), levels=[0.205, 0.22, 0.25, 0.3, 0.4, 0.5, 0.6, 0.68], colors=GREY,
                linewidths=0.8)  # fmt: skip
ax.clabel(cs, fmt=lambda v: num(v, 3), fontsize=7)
for i, (m0, s0) in enumerate(STARTS):
    ax.plot(*path_sat[i].T, color=GREEN, lw=2)
    ax.plot(*path_ns[i].T, color=ORANGE, lw=2, ls="--")
    ax.plot(m0, s0, "o", ms=8, mfc="white", mec="black")
    ax.plot(*path_sat[i, -1], "o", ms=8, color=GREEN)
    ax.plot(*path_ns[i, -1], "s", ms=7, color=ORANGE)
ax.plot([], [], color=GREEN, lw=2, marker="o", label="original loss log(1 − D*) → minimizes the JSD")
ax.plot([], [], color=ORANGE, lw=2, ls="--", marker="s", label="non-saturating −log D* → KL(p_g ‖ p_data) − 2·JSD")
ax.plot([], [], "o", mfc="white", mec="black", label="starting point")
ax.set(xlabel="μ (center of the generator)", ylabel="s (width of the generator)", xlim=(-4, 6), ylim=(0.2, 3))
ax.set_title("Gradient flows with D* recomputed at each step; gray lines: JSD contours (nats)")
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2)
fmt_axes(fig)
plt.show()

The gray lines are contours of the JSD over the $(\mu, s)$ plane; its bottom is at $(0,\, 2.04)$. The solid green
lines (original loss) from four of the six starting points run right down to that bottom; the other two start near
one cluster and stop at that cluster, a local minimum of the JSD. The dashed orange lines (non-saturating) end at one
cluster from five of the six starting points (the squares at $\mu = \pm 2$, $s = 0.5$). Both are "the best single
Gaussian" in some sense, but the loss that everyone uses in practice leans toward **picking one mode and drawing it
very sharply**, rather than covering all of the data. That is the seed of mode collapse.

This is still the idealized game: $D$ always equals $D^*$, and the generator has only two parameters. Real networks
have only an approximate $D$, updated in alternation with $G$, and everything gets more complicated.

## 5 · A two-dimensional GAN on a ring of 8 clusters

Now both $G$ and $D$ are real networks, updated in alternation, as in the demo's second card. The data are 8 very
narrow Gaussian clusters (standard deviation 0.02) spaced evenly on a circle of radius 2, a classic test for mode
collapse (Metz et al., 2016). The architecture and training recipe are the same as in the demo's `viz/gan2d.ts`:
- $G$: $z$ (16 dimensions) → ReLU 32 → ReLU 32 → 2; $D$: $x$ (2 dimensions) → ReLU 32 → ReLU 32 → 1 logit;
- each step: one step for $D$ on a real batch and a fresh fake batch (64 samples of each), then one step for $G$ on
  fresh noise, against the just-updated $D$; Adam with $\eta = 0.002$, $\beta_1 = 0.5$ (the demo stores both
  $\eta = 0.001$ and $0.002$; with 4,000 steps, $0.002$ gets further).

To count the clusters covered, we use the metric of VEEGAN (Srivastava et al., 2017), as the demo does: a sample is
**good** if it lies within 3 standard deviations of the nearest cluster; a cluster is **covered** if it has at least
one good sample, and **gets its share** if it receives at least ¼ of its fair share (1,000 samples / 8 clusters / 4
≈ 31 samples). Every measurement uses the same 1,000 fixed noise vectors.

In [ ]:
MODES = 2 * np.c_[np.cos(np.arange(8) * np.pi / 4), np.sin(np.arange(8) * np.pi / 4)]  # 8 centers on a radius-2 circle
SD = 0.02


def ring_data(rng, R, B):
    """R × B real samples: pick a cluster uniformly at random, add Gaussian noise with standard deviation 0.02."""
    return MODES[rng.integers(0, 8, (R, B))] + SD * rng.standard_normal((R, B, 2))


def score(x):
    """x: R × n × 2. The number of good samples in each cluster (R × 8) and the fraction of good samples of each run."""
    d2 = ((x[:, :, None, :] - MODES) ** 2).sum(-1)
    k, good = d2.argmin(-1), np.sqrt(d2.min(-1)) <= 3 * SD
    counts = np.stack([np.bincount(k[r][good[r]], minlength=8) for r in range(len(x))])
    return counts, good.mean(1)


c, q = score(ring_data(np.random.default_rng(1), 1, 1000))
print(
    f"1,000 real samples: {(c > 0).sum()} clusters covered, {(c >= 1000 / 32).sum()} with their share, {pct(q[0], 1)} good samples"
)

### The R1 gradient penalty

We will compare four configurations: the two losses of $G$ (section 3), each with and without the **R1 gradient
penalty** (Mescheder et al., 2018). R1 adds to $D$'s loss a term that penalizes the slope of the logit $a(x)$ **at
the real data**:

$$R_1 = \frac{\gamma}{2}\, \mathbb E_{x \sim p_{\text{data}}} \big[\lVert \nabla_x a(x) \rVert^2\big]$$

When $G$ already matches the data, an unpenalized $D$ can still be steep around each real point and push $G$ away
from the right place; $R_1$ forces $D$ to be flat there, and Mescheder et al. prove that it makes training converge
locally, whereas unregularized training can circle around the equilibrium forever.

With $D$ a ReLU network, in a region where the ReLU masks $m_1, m_2$ are fixed, $a(x)$ is linear in $x$, so
$\nabla_x a = u_1 W_0^{\top}$ with $u_2 = m_2 \odot W_2^{\top}$ and $u_1 = m_1 \odot (u_2 W_1^{\top})$ (row
convention). The derivative of $R_1$ with respect to the weight matrices runs back along that same chain; the biases
$b$ get no gradient at all. The cell below writes $R_1$, then checks it and the backpropagation of both networks
with finite differences on two tiny networks.

In [ ]:
def r1_grad(D, hs, gamma):
    """R1 on the real samples already run through D (hs = D.fwd(x)), for D = 2 → H → H → 1: the flat gradient and the
    value. gamma: an R × 1 × 1 array (one γ per run)."""
    W0, W1, W2 = D.p[0], D.p[2], D.p[4]
    m1, m2 = hs[1] > 0, hs[2] > 0
    u2 = m2 * W2.transpose(0, 2, 1)  # R × B × H
    u1 = m1 * (u2 @ W1.transpose(0, 2, 1))
    gx = u1 @ W0.transpose(0, 2, 1)  # ∇ₓa(x): R × B × 2
    dgx = gamma * gx / gx.shape[1]  # ∂R1/∂(∇ₓa)
    e = m1 * (dgx @ W0)
    gW = [dgx.transpose(0, 2, 1) @ u1, e.transpose(0, 2, 1) @ u2, ((e @ W1) * m2).sum(1)[:, :, None]]
    flat = [gW[0], np.zeros_like(D.p[1]), gW[1], np.zeros_like(D.p[3]), gW[2], np.zeros_like(D.p[5])]
    return np.concatenate([q.ravel() for q in flat]), 0.5 * gamma[:, 0, 0] * (gx**2).sum(2).mean(1)


def fd_worst(net, f, g, rng, k=12, h=1e-6):
    """The largest relative error between the gradient g and finite differences of f() on k random parameters of net."""
    worst = 0.0
    for i in rng.choice(net.flat.size, k, replace=False):
        old = net.flat[i]
        net.flat[i] = old + h
        fp = f()
        net.flat[i] = old - h
        fm = f()
        net.flat[i] = old
        num = (fp - fm) / (2 * h)
        worst = max(worst, abs(num - g[i]) / max(1e-7, abs(num) + abs(g[i])))
    return worst


rng = np.random.default_rng(3)
Gt, Dt = MLP([3, 5, 4, 2], 1, rng), MLP([2, 6, 5, 1], 1, rng)
Dt.flat += 0.1 * rng.standard_normal(Dt.flat.size)  # nonzero biases, so that the ReLU masks aren't trivial
xr_t, z_t, gam_t = rng.standard_normal((1, 7, 2)), rng.standard_normal((1, 7, 3)), np.full((1, 1, 1), 0.7)
gR1 = r1_grad(Dt, Dt.fwd(xr_t), gam_t)[0]
err_r1 = fd_worst(Dt, lambda: r1_grad(Dt, Dt.fwd(xr_t), gam_t)[1][0], gR1, rng)


def g_loss_ns():
    return softplus(-Dt.fwd(Gt.fwd(z_t)[-1])[-1]).mean()


hg = Gt.fwd(z_t)
hf = Dt.fwd(hg[-1])
_, dx = Dt.back(hf, -sigmoid(-hf[-1]) / 7, params=False)
err_g = fd_worst(Gt, g_loss_ns, Gt.back(hg, dx)[0], rng)
print(f"largest relative error: R1 w.r.t. D's weights {sci(err_r1)} · G's loss through D {sci(err_g)}")
assert err_r1 < 1e-5 and err_g < 1e-5

In [ ]:
def train_ring(losses, gammas, steps, lr=2e-3, dz=16, H=32, B=64, seed=0, every=250):
    """Train len(losses) GANs at once (each run with one loss, 'ns'/'sat', and one γ). Every `every` steps, record the
    number of good samples in each cluster on 1,000 fixed noise vectors, along with the mean D(real) and D(fake) of the
    batch just learned from."""
    R = len(losses)
    rng = np.random.default_rng(seed)
    G, D = MLP([dz, H, H, 2], R, rng), MLP([2, H, H, 1], R, rng)
    ns = np.array([loss == "ns" for loss in losses], float)[:, None, None]
    gam = np.asarray(gammas, float)[:, None, None]
    z_eval = np.broadcast_to(np.random.default_rng(12345).standard_normal((1000, dz)), (R, 1000, dz))
    log = []
    for s in range(steps + 1):
        if s % every == 0:
            counts, good = score(G.fwd(z_eval)[-1])
            log.append({"step": s, "counts": counts, "good": good})
        if s == steps:
            break
        # D's step: cross-entropy (real label 1, fake label 0), plus R1 on the real samples
        xr, xf = ring_data(rng, R, B), G.fwd(rng.standard_normal((R, B, dz)))[-1]
        hs = D.fwd(np.concatenate([xr, xf], 1))
        a = hs[-1]
        gD, _ = D.back(hs, np.concatenate([-sigmoid(-a[:, :B]), sigmoid(a[:, B:])], 1) / B)
        if gam.any():
            gD += r1_grad(D, [h[:, :B] for h in hs], gam)[0]
        D.adam(gD, lr)
        # G's step on fresh noise, against the just-updated D; the gradient goes through D (∂/∂x) and then into G
        hg = G.fwd(rng.standard_normal((R, B, dz)))
        hf = D.fwd(hg[-1])
        af = hf[-1]
        _, dx = D.back(hf, -(ns * sigmoid(-af) + (1 - ns) * sigmoid(af)) / B, params=False)
        G.adam(G.back(hg, dx)[0], lr)
        if (s + 1) % every == 0:
            log.append({"dreal": sigmoid(a[:, :B]).mean((1, 2)), "dfake": sigmoid(a[:, B:]).mean((1, 2))})
    return G, D, [e for e in log if "step" in e], [e for e in log if "dreal" in e]


CONFIGS = [("ns", 0.0), ("sat", 0.0), ("ns", 0.1), ("sat", 0.1)]  # (G's loss, R1's γ)
NAMES = ["non-saturating, no R1", "original loss, no R1", "non-saturating, R1 γ = 0.1", "original loss, R1 γ = 0.1"]
SEEDS = 3

### 🤔 Predict first
The cell below trains 4 configurations × 3 runs (12 GANs in parallel) for 4,000 steps. Without R1 and with the
non-saturating loss, how many of the 8 clusters does the generator cover? And at the end, is $D(\text{real})$ close
to ½?

In [ ]:
G2, D2, ring_log, d_log = train_ring([c[0] for c in CONFIGS for _ in range(SEEDS)],
                                     [c[1] for c in CONFIGS for _ in range(SEEDS)], 4000)  # fmt: skip
final = ring_log[-1]
modes, fair = (final["counts"] > 0).sum(1), (final["counts"] >= 1000 / 32).sum(1)
dreal = np.mean([e["dreal"] for e in d_log[-4:]], 0)  # mean over the last 1,000 steps
dfake = np.mean([e["dfake"] for e in d_log[-4:]], 0)
for i, name in enumerate(NAMES):
    r = slice(SEEDS * i, SEEDS * (i + 1))
    print(f"{name:>26}: clusters covered {modes[r].tolist()} · with their share {fair[r].tolist()} · "
          f"good samples {span(final['good'][r] * 100, 0)}% · D(real) {span(dreal[r])} · D(fake) {span(dfake[r])}")  # fmt: skip
# Check the overall picture rather than each individual number: a GAN amplifies rounding errors, so on a different CPU
# (Colab is x86) an individual run can take a different path. The thresholds below were tried with 4 seeds ×
# float32/float64.
no_r1, with_r1, sat_no = slice(0, 2 * SEEDS), slice(2 * SEEDS, 4 * SEEDS), slice(SEEDS, 2 * SEEDS)
assert modes[no_r1].mean() <= 4 and modes[with_r1].mean() >= 6.5, "R1 must cover clearly more clusters"
assert final["good"][sat_no].mean() < 0.3 and fair[sat_no].max() <= 2, "the original loss without R1 must get stuck"
assert dreal[no_r1].mean() > 0.7 and abs(dreal[with_r1] - 0.5).max() < 0.08, "D wins without R1, ≈ ½ with it"

The figure below plots the first run of each configuration. Top row: the generator's 1,000 samples at the last step
(orange squares) over the real data (blue dots). Bottom row: over time (horizontal axis), each row is a cluster, and
the darker a cell, the more good samples that cluster has.

In [ ]:
real_pts = ring_data(np.random.default_rng(7), 1, 300)[0]
x_end = G2.fwd(np.random.default_rng(12345).standard_normal((G2.p[0].shape[0], 1000, 16)))[-1]
steps_logged = np.array([e["step"] for e in ring_log])
counts_t = np.stack([e["counts"] for e in ring_log], 2)  # run × cluster × checkpoint
fig, axes = plt.subplots(2, 4, figsize=(10, 5.2), gridspec_kw={"height_ratios": [1.25, 1]})
for i, name in enumerate(NAMES):
    r = SEEDS * i  # the first run of configuration i
    ax = axes[0, i]
    ax.scatter(*real_pts.T, s=6, color=BLUE, marker="o", label="real data")
    ax.scatter(*x_end[r].T, s=4, color=ORANGE, marker="s", alpha=0.5, label="samples of G")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", xticks=[-2, 0, 2], yticks=[-2, 0, 2])
    ax.set_title(
        f"{name.replace(', ', chr(10))}\n{modes[r]} cluster{'' if modes[r] == 1 else 's'} covered, {fair[r]} with share",
        fontsize=9,
    )
    ax = axes[1, i]
    ax.imshow(counts_t[r] / 1000, aspect="auto", cmap="Oranges", vmin=0, vmax=0.25, interpolation="nearest",
              extent=[0, steps_logged[-1], 7.5, -0.5])  # fmt: skip
    ax.set(xlabel="step", yticks=range(8), yticklabels=[str(k) for k in range(8)] if i == 0 else [])
    ax.set_xticks([0, 2000, 4000])
axes[1, 0].set_ylabel("cluster")
axes[0, 0].legend(frameon=False, fontsize=7, loc="lower left", markerscale=2)
plt.tight_layout()
fmt_axes(fig)
plt.show()

Without R1, the generator with the non-saturating loss quickly piles its samples onto 1–3 clusters and stays there:
the other rows of the bottom plot stay almost white throughout training. That is **mode collapse**: many different
$z$ map to the same place. Just as in the one-dimensional game of section 4: the non-saturating loss accepts
missing clusters, as long as every sample looks real.

The original loss without R1 is even worse: most samples land in no cluster at all. $D$ wins too easily (D(real)
about 0.9, D(fake) about 0.1 in the runs above), and as in section 3, the gradient of the original loss dies out
when $D$ rejects easily.

With R1, both losses spread their samples all around the ring, and $D(\text{real}) \approx D(\text{fake}) \approx
\tfrac12$. After 4,000 steps, the samples haven't yet gathered tightly into the clusters (many samples lie on the
arc between two clusters), so the number of clusters that "get their share" is usually still low and varies a lot
from run to run; the demo's longer runs show what happens next.

### Compared with the demo's stored runs
`gan-2d.json` stores this same recipe, run with the demo's TypeScript code (`viz/gan2d.ts`) with 5 seeds and 20,000
steps, the clusters counted every 500 steps on 2,000 samples. Its random numbers are entirely different from ours,
so only a qualitative comparison is possible. The cell below reads the cluster counts at step 4,000 (to compare with
our run) and at step 20,000.

In [ ]:
def checkpoints(run):
    """The checkpoints (every 500 steps) of a stored run: clusters covered, clusters with their share, fraction of good
    samples, most populated cluster."""
    b = np.frombuffer(base64.b64decode(run["traj"]), np.uint8).reshape(-1, 4).astype(int)
    return {"modes": b[:, 0], "fair": b[:, 1], "good": b[:, 2] / 250, "top": b[:, 3] - 1}


at4k = 4000 // two["setup"]["every"]
for (loss, g), name in zip(CONFIGS, NAMES):
    runs = two["runs"][f"ring-H32-{loss}-dz16-r{g:g}-alt-lr0.002"]["seeds"]
    cp = [checkpoints(r) for r in runs]
    print(f"{name:>26} (5 seeds): step 4,000: clusters covered {[int(c['modes'][at4k]) for c in cp]} · "
          f"step 20,000: covered {[r['final']['modes'] for r in runs]}, with their share {[r['final']['fair'] for r in runs]}, "
          f"D(real) {span([r['final']['dReal'] for r in runs])}")  # fmt: skip

The same picture: at step 4,000, 1–5 clusters are covered without R1, and all 8 in every seed with R1. After 20,000
steps, R1 gathers the samples into the clusters: with the non-saturating loss, 7–8 clusters get their share in all
5 seeds (original loss: 3–8). Without R1, after 20,000 steps, still only 1–4 clusters are covered. More steps don't
cure mode collapse.

### Mode hopping, or just noise in the metric?
Metz et al. (2016) describe a kind of collapse that doesn't stand still: the generator piles onto one cluster, $D$
learns to reject that cluster, the generator jumps to another, and the chase never ends. One way to measure it is to
count how many times the "most populated cluster" changes. But that measure only means something when there are
enough good samples to count. The cell below looks at the stored runs with the original loss and a hidden layer of
64, where the "most populated cluster" changes the most in the second half of training.

In [ ]:
hop = two["runs"]["ring-H64-sat-dz16-r0-alt-lr0.001"]["seeds"]
for r in hop:
    cp = checkpoints(r)
    half = slice(len(cp["top"]) // 2, None)
    top = cp["top"][half][cp["top"][half] >= 0]
    print(f"seed {r['seed']}: the most populated cluster changes {int((top[1:] != top[:-1]).sum())} times, through "
          f"{len(set(top))} clusters · but at most only {pct(cp['good'][half].max(), 1)} of the 2,000 samples are good")  # fmt: skip
pts = np.frombuffer(base64.b64decode(hop[0]["pts"]), "<i2").reshape(-1, 2) / 1000  # the last 300 samples of seed 0
d_mode = np.sqrt(((pts[:, None] - MODES) ** 2).sum(-1)).min(1)
print(f"seed 0, last 300 samples: median distance to the nearest cluster {num(np.median(d_mode), 2)}, i.e. "
      f"{num(np.median(d_mode) / SD, 0)} standard deviations of a cluster; radius from {num(np.hypot(*pts.T).min(), 1)} "
      f"to {num(np.hypot(*pts.T).max(), 1)}")  # fmt: skip

In all 3 seeds, the "most populated cluster" changes 15–16 times and moves through 7 clusters. But almost no sample
lies within a cluster: at most 0.4–0.8% of the 2,000 samples, that is, 8–16 samples spread over 8 clusters, so the
"most populated cluster" changes with the noise. The last 300 samples of seed 0 are scattered over a wide region,
with a median distance to the nearest cluster of about 50 standard deviations. That is not a generator hopping from
one cluster to another, but a generator that has broken down completely (original loss, $D$ wins outright).

What about the runs that really collapse? The cell below picks every stored run on the ring of 8 clusters where, at
at least 70% of the checkpoints in the second half, at least 30% of the samples are good and only 1–3 clusters get
their share, then counts how many clusters their most populated cluster moves through.

In [ ]:
visited = []
for run in two["runs"].values():
    for r in run["seeds"] if run["cfg"]["data"] == "ring" else []:
        cp = checkpoints(r)
        half = slice(len(cp["top"]) // 2, None)
        ok = (cp["good"][half] >= 0.3) & (cp["fair"][half] <= 3)
        if r["final"] and ok.mean() >= 0.7:
            visited.append(len(set(cp["top"][half][ok])))
print(f"{len(visited)} runs that really collapse; in the second half, the most populated cluster moves through: "
      f"1 cluster in {visited.count(1)} runs, 2 clusters in {visited.count(2)} runs, more in {sum(v > 2 for v in visited)} runs")  # fmt: skip

In every run that really collapses, the most populated cluster stays put or only switches back and forth between 2
clusters. In these runs, the generator doesn't "hop around" the clusters as Metz et al. (2016) describe; it stops at
one or two clusters and stays there.

The general lesson: a metric can be trusted only when its conditions are met. Section 8 meets this again with
$D \approx \tfrac12$.

## 6 · Digits from noise: the demo's MNIST generator

On MNIST, $G$ has exactly the architecture of Lesson 19's decoder: $z$ (16 dimensions) → ReLU 128 → ReLU 256 → 784
→ sigmoid, and $D$ mirrors the encoder: 784 → LeakyReLU 256 → LeakyReLU 128 → 1 logit. The two networks learn with
exactly the loop of section 5 (Adam $\beta_1 = 0.5$, batch 64, 20,000 steps) on 55,000 grayscale images. One
training run takes many minutes on a CPU, so we load the best generator of the grid (non-saturating loss, R1 γ = 10,
η = 0.001, seed 0) from `gan-g16.json`, in the same format as Lesson 19: hidden layers in float16, the output
layer's matrix in int8 with one float16 scale per column.

The cell below unpacks it, generates 10,000 digits from the noise `default_rng(7)` (exactly the noise stream that
the demo's measurements use), then compares the first 24 images with the 24 images the demo stored (generated with
the unrounded weights).

In [ ]:
def b64(s, dtype):
    """Decode an array that the demo stored as base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float32).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float32)  # one scale per column
    return b64(t["i8"], np.int8).astype(np.float32).reshape(t["shape"]) * scale


Gm = {k: unpack(t) for k, t in g16["dec"].items()}


def generate(z):
    """G(z): 16 → ReLU 128 → ReLU 256 → 784 → sigmoid; each row of the result is an image, pixels in [0, 1]."""
    h1 = np.maximum(z @ Gm["d0.W"] + Gm["d0.b"], 0)
    h2 = np.maximum(h1 @ Gm["d1.W"] + Gm["d1.b"], 0)
    return sigmoid(h2 @ Gm["out.W"] + Gm["out.b"])


def u8(s):
    """Images the demo stored as base64 uint8 → an n × 784 array in [0, 1]."""
    return b64(s, np.uint8).reshape(-1, 784).astype(np.float32) / 255


Xg = generate(np.random.default_rng(7).standard_normal((10000, 16)).astype(np.float32))
n_par = sum(v.size for v in Gm.values())
diff = np.abs(Xg[:24] - u8(mn["images"]["ns-r10-lr0.001"]))
print({k: v.shape for k, v in Gm.items() if k.endswith(".W")}, f"· {num(n_par, 0)} parameters")
print(
    f"first 24 images vs. the images the demo stored: mean difference {num(diff.mean(), 4)}, largest {num(diff.max(), 3)} (on a 0–1 scale)"
)
assert n_par == 236688 and diff.mean() < 0.002 and diff.max() < 0.06, "differs from the demo: check the generator again"

The largest difference is a few percent of the pixel scale: that is the int8/float16 rounding plus the rounding of
the stored images to uint8. The figure below shows the first 48 samples, and the last row is 9 images on the straight
line in noise space from sample 1 to sample 2 (the first two images, at the top left).

In [ ]:
z_ab = np.random.default_rng(7).standard_normal((2, 16)).astype(np.float32)
frames = generate(z_ab[0] + np.linspace(0, 1, 9)[:, None] * (z_ab[1] - z_ab[0]))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), gridspec_kw={"height_ratios": [4, 1.1]})
a1.imshow(tile(Xg[:48], 4, 12), cmap="gray_r", vmin=0, vmax=1)
a1.set_title("48 digits the generator makes from noise z ~ N(0, I) (R1 γ = 10, seed 0)", fontsize=10)
a2.imshow(tile(frames, 1, 9), cmap="gray_r", vmin=0, vmax=1)
a2.set_title("straight-line interpolation in noise space, from sample 1 to sample 2", fontsize=10)
for ax in (a1, a2):
    ax.axis("off")
plt.tight_layout()
plt.show()

Most samples read as digits, with strokes more continuous and sharper than the VAE samples of Lesson 19, although
some images are still distorted. Walking along the straight line between two noise vectors, the image changes
continuously from one digit into another: the generator is a smooth function of $z$.

### 🤔 Predict first
The generator **never** receives a real image: all the information about the data reaches it through the gradients
of $D$ (Goodfellow et al., 2014, section 6). So can its samples be copies of training images? Measure with the pixel
distance (the Euclidean norm over the 784 pixels) from each sample to the nearest training image: are the GAN's
samples **closer** to the training images or **farther** from them than a real test image (a digit the model hasn't
seen)?

In [ ]:
def nearest(A, T, bs=250):
    """The pixel distance from each row of A to the nearest row of T, and the index of that row."""
    tn = (T.astype(np.float64) ** 2).sum(1)
    dist, idx = np.empty(len(A)), np.empty(len(A), np.int64)
    for i in range(0, len(A), bs):
        x = A[i : i + bs]
        d2 = (x.astype(np.float64) ** 2).sum(1)[:, None] + tn - 2 * (x @ T.T).astype(np.float64)
        idx[i : i + bs] = d2.argmin(1)
        dist[i : i + bs] = np.sqrt(np.maximum(d2[np.arange(len(x)), idx[i : i + bs]], 0))
    return dist, idx


d_gan, i_gan = nearest(Xg[:1000], Xtr)  # 1,000 samples → 55,000 training images
d_test, _ = nearest(Xte[:1000], Xtr)  # 1,000 real test images → 55,000 training images
QS = (0.01, 0.1, 0.5)
thr = np.quantile(d_test, 0.01)  # "copying": closer to a training image than 99% of test images
copy = (d_gan < thr).mean()
print("1% · 10% · 50% quantiles of the distance to the nearest training image:")
print("  GAN samples:", " · ".join(num(v, 2) for v in np.quantile(d_gan, QS)))
print("  test images:", " · ".join(num(v, 2) for v in np.quantile(d_test, QS)))
print(f"samples closer to a training image than 99% of test images (below {num(thr, 2)}): {pct(copy, 1)}")
ref = mn["runs"]["ns-r10-lr0.001"]["seeds"][0]["nn"]
assert np.allclose(np.quantile(d_test, QS), ref["test"], atol=2e-3), "differs from the demo: check the data again"
assert np.allclose(np.quantile(d_gan, QS), ref["sample"], atol=0.03) and copy <= 0.002

**Farther.** The real test images lie closer to the training images than the GAN's samples do at every quantile,
and none of the 1,000 samples is closer to a training image than 99% of test images are. With 55,000 images, this
generator doesn't copy: its samples deviate a little from real digits (distorted strokes, odd thicknesses), so they
are farther from every training image than a new real digit is.

But "never receiving a real image directly" doesn't prevent copying. $D$ learns on real images, and if the training
set is small, $D$ memorizes it and then teaches it back to $G$. The demo trains the same configuration on the first
100 and the first 1,000 images. The cell below reads their copy rates, then recomputes the distance to the nearest
training image for the 8 samples that the demo stored from the GAN trained on 100 images.

In [ ]:
for name in ("n100", "n1000", "ns-r10-lr0.001"):
    seeds = mn["runs"][name]["seeds"]
    n = mn["runs"][name]["cfg"]["ntrain"]
    print(f"trained on {num(n, 0):>6} images: {span([100 * s['copy'] for s in seeds], 1)}% of samples are copies (3 seeds) · "
          f"threshold: 1% of test images are closer than {num(seeds[0]['nn']['test'][0], 2)}")  # fmt: skip
d100, _ = nearest(Xte[:1000], Xtr[:100])  # test images → 100 training images
assert np.allclose(np.quantile(d100, QS), mn["runs"]["n100"]["seeds"][0]["nn"]["test"], atol=2e-3)
x8, near8 = u8(mn["nn"]["n100"]["x"]), u8(mn["nn"]["n100"]["near"])
d8, i8 = nearest(x8, Xtr[:100])
print("8 samples (GAN trained on 100 images) → nearest training image:", " · ".join(num(v, 2) for v in d8))
assert np.allclose(d8, mn["nn"]["n100"]["dist"], atol=0.01) and np.abs(Xtr[:100][i8] - near8).max() < 0.003
print(
    f"copy threshold with 100 images: {num(np.quantile(d100, 0.01), 2)}; {(d8 < np.quantile(d100, 0.01)).sum()}/8 samples below the threshold"
)

fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for k in range(8):
    axes[0, k].imshow(x8[k].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    axes[1, k].imshow(Xtr[:100][i8[k]].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    axes[1, k].set_title(f"distance {num(d8[k], 1)}", fontsize=8)
    for a in axes[:, k]:
        a.set_xticks([])
        a.set_yticks([])
axes[0, 0].set_ylabel("sample", fontsize=9)
axes[1, 0].set_ylabel("nearest\ntraining image", fontsize=9)
plt.tight_layout()
plt.show()

The top row is 8 samples of the GAN trained on 100 images, the bottom row the nearest training image of each sample,
with its distance. Most pairs are nearly identical. With 100 images, about half of the samples are closer to a
training image than 99% of test images are; with 1,000 images, nearly none; with 55,000 images, none. Carlini et al.
(2023) also extracted training images from large GANs on CIFAR-10, though fewer than from diffusion models of the
same quality.

## 7 · The experiment grid on MNIST: G's loss and R1

The demo trains a grid of configurations, 3 seeds per configuration, with the same two networks as in section 6.
Each run is measured on 10,000 samples with Lesson 19's metric: **FD**, the Fréchet distance between the LeNet
features of the samples and those of the 10,000 test images (the FID recipe, with Lesson 12's LeNet in place of
Inception, so FD values can only be compared within the course). Two sets of real images (5,000 validation images
against the 10,000 test images) give an FD of 1.32. The cell below prints the whole grid; "failed" means an FD
above 100 (the images are no longer digits).

In [ ]:
def label(name):
    c = mn["runs"][name]["cfg"]
    parts = ["non-saturating" if c["loss"] == "ns" else "original loss", f"R1 γ {c['r1']:g}" if c["r1"] else "no R1"]
    parts.append(f"η {c['lr']:g}")
    parts += [f"d = {c['d']}"] * (c["d"] != 16) + [f"{num(c['ntrain'], 0)} images"] * (c["ntrain"] < 55000)
    return ", ".join(parts + [f"β₁ {num(c['b1'], 1)}"] * (c["b1"] != 0.5))


def fds(name):
    return [s["fd"] for s in mn["runs"][name]["seeds"] if not s.get("diverged")]


def failed(seeds):
    return sum(s.get("diverged", False) or s["fd"] > 100 for s in seeds)


print(f"{'configuration':>46} {'FD (3 seeds)':>14} {'precision':>11} {'recall':>11} {'failed':>6}")
for name, r in mn["runs"].items():
    s = r["seeds"]
    print(f"{label(name):>46} {span(fds(name)):>14} {span([x['P'] for x in s]):>11} {span([x['R'] for x in s]):>11}"
          f" {failed(s)}/{len(s):>2}")  # fmt: skip
print(f"{'Lesson 19 VAE d = 16, z ~ N(0, I)':>46} {span([x['fd'] for x in mn['vae']['prior']['seeds']]):>14}")
print(f"{'Lesson 19 VAE d = 16, GMM fitted to the codes':>46} {span([x['fd'] for x in mn['vae']['gmm']['seeds']]):>14}")
print(f"{'two sets of real images (5,000 validation)':>46} {num(mn['judge']['floor']):>14}")

Counting only the runs on exactly the pair of networks of section 6 (16-dimensional noise, 55,000 images,
$\beta_1 = 0.5$), which differ only in $G$'s loss, $\eta$ and R1:

In [ ]:
same = [
    n for n, r in mn["runs"].items() if r["cfg"]["d"] == 16 and r["cfg"]["ntrain"] == 55000 and r["cfg"]["b1"] == 0.5
]
for has_r1 in (False, True):
    names = [n for n in same if (mn["runs"][n]["cfg"]["r1"] > 0) == has_r1]
    seeds = [s for n in names for s in mn["runs"][n]["seeds"]]
    bad = [label(n) for n in names for s in mn["runs"][n]["seeds"] if failed([s])]
    print(f"{'with R1' if has_r1 else 'no R1'}: {failed(seeds)}/{len(seeds)} runs failed · FD "
          f"{span([s['fd'] for s in seeds])}" + (f" · failed: {'; '.join(sorted(set(bad)))}" if bad else ""))  # fmt: skip
best = fds("ns-r10-lr0.001")
print(f"best (non-saturating, R1 γ 10, η 0.001): FD {span(best)}; Lesson 19's VAE + GMM: "
      f"{span([x['fd'] for x in mn['vae']['gmm']['seeds']])}")  # fmt: skip

Without R1, 4/12 runs fail, and all 4 use the original loss: at $\eta = 0.001$ all 3 seeds get stuck at FD 288–317,
the very generator whose gradient section 3 found to be 14,000 times smaller. The non-saturating loss never fails,
but without R1 it stops at FD 23–33. With R1, 0/18 runs fail, even with the original loss; FD comes down to 7–11
with γ 1 and 10 (γ = 100 penalizes too strongly: 16–17). The gradient penalty makes most of the "hard to train"
go away. The best generator has an FD of 7.88–8.02, on par with Lesson 19's VAE + GMM (7.88–8.94; within the
variation between seeds), and clearly better than the VAE that draws $z$ from $\mathcal N(0, I)$ (31.18–31.82), even
though the decoder architecture is the same.

Two other rows are worth noting. $\beta_1 = 0.9$ (Adam's default) without R1 fails in all 3 seeds: Radford et al.
(2015) write that leaving $\beta_1$ at 0.9 caused oscillation and instability, and that lowering it to 0.5 helped
stabilize training. Noise with only 2 dimensions ($d = 2$) gives FD 20–31 and a recall near 0: two numbers aren't
enough to produce every style of handwriting.

The figure below plots FD (log axis) against the training step for 4 configurations, one line per seed, together
with $D(G(z))$, the probability that $D$ at that moment gives to a fake being real.

In [ ]:
SHOW = [("ns-r10-lr0.001", BLUE, "-"), ("ns-r0-lr0.001", ORANGE, "--"), ("sat-r0-lr0.001", GREEN, ":"),
        ("sat-r10-lr0.001", GREY, "-.")]  # fmt: skip
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for name, color, ls in SHOW:
    for k, s in enumerate(mn["runs"][name]["seeds"]):
        t_fd = [(t["step"], t["fd"]) for t in s["traj"] if "fd" in t]
        t_d = [(t["step"], t["DGz"]) for t in s["traj"] if t["step"] > 0]
        a1.plot(*zip(*t_fd), color=color, ls=ls, lw=1.6, label=label(name) if k == 0 else None)
        a2.plot(*zip(*t_d), color=color, ls=ls, lw=1.6)
a1.axhline(mn["judge"]["floor"], color="black", lw=0.8)
a1.text(20000, mn["judge"]["floor"] * 1.15, "two sets of real images (5,000 validation)", ha="right", fontsize=8)
a1.set(
    yscale="log",
    ylim=(1, 500),
    xlabel="step",
    ylabel="FD (lower is better)",
    title="FD of 5,000 samples, every 1,000 steps",
)
a2.set(
    xscale="log", yscale="log", ylim=(1e-6, 1), xlabel="step", ylabel="D(G(z))", title="D(fake image) during training"
)
a2.axhline(0.5, color="black", lw=0.8)
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2)
fmt_axes(fig)
plt.show()
ld = [t["LD"] for s in mn["runs"]["ns-r10-lr0.001"]["seeds"] for t in s["traj"] if t["step"] >= 2000]
print(f"best: D's loss from step 2,000 to the end stays within {span(ld)} (chance: 2·ln 2 = {num(2 * math.log(2))})")
fd_at = {k: [next(t["fd"] for t in s["traj"] if t["step"] == k) for s in mn["runs"]["ns-r10-lr0.001"]["seeds"]]
         for k in (0, 2000)}  # fmt: skip
print(f"  while FD goes from {span(fd_at[0], 0)} (step 0) through {span(fd_at[2000], 1)} (step 2,000) down to {span(best)}"
      f" (step 20,000): a further {num(min(fd_at[2000]) / max(best), 1)}–{num(max(fd_at[2000]) / min(best), 1)}-fold drop")  # fmt: skip

The two configurations with R1 (solid blue, dash-dotted gray) lower their FD almost throughout the 20,000 steps,
with some oscillation. The original loss without R1 (dotted green) stays put at an FD near 300: on the right, its
$D(G(z))$ falls below $10^{-2}$ within the first few hundred to 2,000 steps and never comes back up (the file rounds
$D(G(z))$ to $10^{-6}$, so the line breaks at the checkpoints where it is 0). The non-saturating loss without R1
(dashed orange) escapes, but stops at an FD above 20.

More importantly: in the best run, $D$'s loss sits right at $2 \ln 2 \approx 1.39$ (the value for a $D$ that
guesses $\tfrac12$) from step 2,000 to the end, while FD still drops by a further factor of about 6. **The GAN's
loss doesn't tell you whether the images are good yet**: you have to measure from the outside.

## 8 · Evaluation: does $D \approx \tfrac12$ mean success?

At the end of the best run, the GAN's own $D$ gives real images and fake images almost the same probability:

In [ ]:
for name in ("ns-r10-lr0.001", "sat-r0-lr0.001"):
    end = [s["traj"][-1] for s in mn["runs"][name]["seeds"]]
    print(f"{label(name):>32}, step 20,000: D(training images) {span([t['Dtr'] for t in end], 3)} · "
          f"D(unseen images) {span([t['Dva'] for t in end], 3)} · D(G(z)) {span([t['DGz'] for t in end], 3)}")  # fmt: skip

With the best generator, $D$ gives both real and fake images a probability of almost exactly ½. At first glance,
that is the equilibrium of section 4: $D^* = \tfrac12$ when $p_g = p_{\text{data}}$. But section 4.1 already showed
that the converse doesn't hold: the fact that *this run's* $D$ is fooled doesn't mean that *no discriminator* can
tell the two apart.

The direct check is a **classifier two-sample test** (C2ST; Lopez-Paz & Oquab, 2017): train a fresh network, one
that never took part in the game, to tell 5,000 GAN images from 5,000 real images, then measure its accuracy on
another 5,000 + 5,000 images it hasn't seen. 50% means the two can't be told apart. As in the demo, the classifier is
an MLP 784 → 128 → 1 (the `MLP` class of section 4), Adam, batch 128, 10 epochs.

### 🤔 Predict first
The best generator has an FD of 7.88–8.02 and $D \approx \tfrac12$. About what percentage does the fresh classifier
get right: close to 50%, around 70%, or close to 90%? And with two sets of real images (10,000 training images
versus 10,000 test images)?

In [ ]:
def c2st(A, Bset, seed=0, epochs=10, lr=1e-3, bs=128):
    """The accuracy, on the unseen half, of a fresh MLP 784 → 128 → 1 that learns to tell A (label 1) from Bset
    (label 0)."""
    rng = np.random.default_rng(seed)
    n = min(len(A), len(Bset)) // 2
    A, Bset = A[rng.permutation(len(A))], Bset[rng.permutation(len(Bset))]  # shuffle before splitting in half
    Xa, Xb = np.concatenate([A[:n], Bset[:n]]), np.concatenate([A[n : 2 * n], Bset[n : 2 * n]])
    y = np.r_[np.ones(n), np.zeros(n)]
    net = MLP([784, 128, 1], 1, rng)
    for _ in range(epochs):
        order = rng.permutation(2 * n)
        for i in range(0, 2 * n, bs):
            idx = order[i : i + bs]
            hs = net.fwd(Xa[idx][None])
            d_out = (sigmoid(hs[-1][0, :, 0]) - y[idx]) / len(idx)  # binary cross-entropy (Lesson 04)
            net.adam(net.back(hs, d_out[None, :, None])[0], lr, b1=0.9)
    return float(((net.fwd(Xb[None])[-1][0, :, 0] > 0) == (y > 0.5)).mean())


real10k = train_img[np.random.default_rng(50).choice(55000, 10000, replace=False)].astype(np.float32) / 255
acc_gan, acc_real = c2st(Xg, Xte), c2st(real10k, Xte)
print(f"fresh classifier, on unseen images: best GAN (seed 0) vs. real images {pct(acc_gan, 1)} · "
      f"real images vs. real images {pct(acc_real, 1)}")  # fmt: skip
assert 0.85 < acc_gan < 0.92 and abs(acc_real - 0.5323) < 0.01, "differs from the demo: check the data again"

Close to 90%. The fresh classifier tells GAN images from real ones easily, while the two sets of real images give
only about 53%. $D \approx \tfrac12$ only means that the **current** $D$ has been fooled: $D$ and $G$ chase each
other, $D$ is always one step behind, and R1 deliberately keeps $D$ smooth as well. A fresh network, trained to the
end on a fixed problem, finds the differences that $D$ overlooked. (The generator here is the rounded version that
the demo runs; the demo measures on the original weights, so the number differs slightly in the third digit.)

The table below puts C2ST next to FD and precision/recall for a few configurations, from the demo's full runs (3
seeds). **Precision** is the fraction of samples that fall within the region of the real images, **recall** the
fraction of real images that fall within the region of the samples, both on LeNet features with 3 nearest neighbors
(Kynkäänniemi et al., 2019). High precision with low recall is a measurable sign of mode collapse.

In [ ]:
rows = [("two sets of real images (10,000 training)", mn["real"]), ("VAE d = 16, z ~ N(0, I) (Lesson 19)", mn["vae"]["prior"]["seeds"]),
        ("VAE d = 16 + GMM (Lesson 19)", mn["vae"]["gmm"]["seeds"])]  # fmt: skip
rows += [
    (label(n), mn["runs"][n]["seeds"])
    for n in ("ns-r10-lr0.001", "sat-r10-lr0.001", "ns-r0-lr0.001", "d2", "n1000", "n100")
]
print(f"{'':>46} {'FD':>12} {'C2ST':>12} {'precision':>11} {'recall':>11}")
for name, seeds in rows:
    print(f"{name:>46} {span([s['fd'] for s in seeds]):>12} {span([100 * s['c2st'] for s in seeds], 1) + '%':>12} "
          f"{span([s['P'] for s in seeds]):>11} {span([s['R'] for s in seeds]):>11}")  # fmt: skip

Three things can be read from the table:
- **No metric is enough on its own.** Lesson 19's VAE + GMM has an FD on par with the best GAN, but the fresh
  classifier is right only 71–72% of the time on it, versus 88–89% on the GAN: FD looks at LeNet's features, C2ST
  looks straight at the pixels, and each metric sees a different kind of difference. The VAE + GMM's precision is
  higher, the GAN's recall is higher.
- **Low recall is mode collapse, measured.** Noise with 2 dimensions gives a normal precision but a recall near 0:
  the samples look like digits, but in only a few styles.
- **Copying fools precision.** The GAN trained on 100 images has the highest precision of the table's GANs, higher
  even than two sets of real images (its samples *are* the real images, section 6), but a recall of 0.06–0.07 and a
  C2ST of 99%: 100 images don't represent the whole distribution.

### FD depends on the number of samples
FD compares the means and covariances of two sets of features, estimated from samples. With few samples the
covariance estimate is noisier, and FD is **biased upward**: even two sets of real images don't give 0 (Bińkowski et
al., 2018). The cell below reads the demo's measurements (5 draws for each number of samples), then reproduces the
effect with a feature we can compute ourselves: the 32 principal components (PCA, Lesson 19) of the first 10,000
training images in place of LeNet. The numbers differ, the effect is the same.

In [ ]:
for n in mn["fdN"]["real"]:
    print(f"n = {num(int(n), 0):>6}: FD two sets of real images {span(mn['fdN']['real'][n]):>9} · best GAN "
          f"{span(mn['fdN']['gan'][n]):>9} (5 draws, LeNet features, against 10,000 test images)")  # fmt: skip


def fd(F1, F2):
    """The Fréchet distance between two Gaussians fitted to two sets of features: ‖m₁ − m₂‖² + tr(C₁ + C₂ − 2(C₁C₂)^½)."""
    C1, C2 = np.cov(F1, rowvar=False), np.cov(F2, rowvar=False)
    root = np.sqrt(np.clip(np.linalg.eigvals(C1 @ C2).real, 0, None)).sum()  # tr((C₁C₂)^½) via the eigenvalues
    return float(((F1.mean(0) - F2.mean(0)) ** 2).sum() + np.trace(C1) + np.trace(C2) - 2 * root)


P10 = Xtr[:10000].astype(np.float64)
m_pca = P10.mean(0)
B_pca = np.linalg.eigh(np.cov(P10, rowvar=False))[1][:, ::-1][:, :32]  # the 32 directions of largest variance


def feat(X):
    return (X.astype(np.float64) - m_pca) @ B_pca


F_te, F_g = feat(Xte), feat(Xg)
NS = (1000, 2000, 5000, 10000)
fd_real = {n: [fd(feat(Xtr[np.random.default_rng(10 * n + k).choice(55000, n, replace=False)]), F_te) for k in range(3)]
           for n in NS}  # fmt: skip
fd_gan = {
    n: [fd(F_g[np.random.default_rng(n + k).choice(10000, n, replace=False)], F_te) for k in range(3)] for n in NS
}
for n in NS:
    print(
        f"32-dim PCA, n = {num(n, 0):>6}: two sets of real images {span(fd_real[n], 3)} · GAN {span(fd_gan[n], 3)} (3 draws)"
    )
mean_real = [np.mean(fd_real[n]) for n in NS]
lenet = [np.mean(mn["fdN"]["real"][str(n)]) for n in (1000, 10000)]
print(f"two sets of real images, 1,000 → 10,000 samples: FD drops {num(mean_real[0] / mean_real[-1], 1)}-fold (PCA), "
      f"{num(lenet[0] / lenet[1], 1)}-fold (LeNet)")  # fmt: skip
assert all(a > b for a, b in itertools.pairwise(mean_real)) and mean_real[0] > 1.8 * mean_real[-1]

With both kinds of features, the FD of two sets of **real** images drops by a factor of 2.5–3.2 when going from
1,000 to 10,000 samples: that upward bias belongs to the estimate, not to the data. That is why FD values can only be
compared at the same number of samples and with the same feature network. Every FD in this lesson uses 10,000
samples and Lesson 12's LeNet; the FID in papers uses Inception, usually with 50,000 samples (as in StyleGAN, Karras
et al., 2019), so none of the numbers here can be compared with published FIDs.

## 9 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

**The one-dimensional game:** pick your own starting points. Try starting far away ($\mu = -5$), or with a very
large $s$.

In [ ]:
my_starts = [(-5.0, 0.5), (0.0, 0.3), (0.0, 3.0), (2.5, 2.5)]
p_sat, p_ns = flows(my_starts)
for (m0, s0), a_, b_ in zip(my_starts, p_sat[:, -1], p_ns[:, -1]):
    print(f"({num(m0, 1)}, {num(s0, 1)}) → original loss ({num(a_[0], 2)}, {num(a_[1], 2)}), JSD {num(divergences(*a_)[0], 4)}"
          f" · non-saturating ({num(b_[0], 2)}, {num(b_[1], 2)}), JSD {num(divergences(*b_)[0], 4)}")  # fmt: skip

**The ring of 8 clusters with 2-dimensional noise:** the demo shows that the dimension of the noise changes the
picture entirely. The cell below runs the non-saturating, no-R1 configuration of section 5, changing only $z$ from
16 to 2 dimensions (3 runs, 3,000 steps), then prints the demo's stored results for the same configuration. Also try
`gammas=[1.0] * 3` or `losses=["sat"] * 3`.

In [ ]:
_, _, log_dz2, _ = train_ring(["ns"] * 3, [0.0] * 3, 3000, dz=2, every=3000)
c_dz2 = log_dz2[-1]["counts"]
print(f"2-dimensional noise, no R1, 3,000 steps: clusters covered {(c_dz2 > 0).sum(1).tolist()} · with their share "
      f"{(c_dz2 >= 1000 / 32).sum(1).tolist()} · good samples {span(log_dz2[-1]['good'] * 100, 0)}%")  # fmt: skip
st = two["runs"]["ring-H32-ns-dz2-r0-alt-lr0.002"]["seeds"]
print(f"demo, 5 seeds × 20,000 steps: clusters covered {[r['final']['modes'] for r in st]} · with their share "
      f"{[r['final']['fair'] for r in st]} · good samples {span([100 * r['final']['hq'] for r in st], 0)}%")  # fmt: skip

With 2-dimensional noise, the generator spreads its samples over clearly more clusters than with 16-dimensional
noise, even without R1. That is an empirical result on this data, not a law: on MNIST it is the other way around,
and 2-dimensional noise gives a recall near 0 (section 8), because 2 numbers aren't enough to describe 10 digits in
every handwriting style. One thing holds for any dimension: $G$ is a continuous function of $z$, so its set of
samples is all in one piece, and the generator cannot put samples in 8 separate clusters without some samples lying
between them.

## 10 · Limits and next steps

- **There is no loss function to monitor.** $D$'s loss stays put around $2\ln 2$ while the images still get several
  times better; to know how good the model is, you have to measure from the outside, with several metrics.
- **Training is a game, not an optimization problem.** With finite networks, a Nash equilibrium may not exist
  (Farnia & Ozdaglar, 2020); gradient penalties like R1 make training converge locally (Mescheder et al., 2018), but
  don't guarantee covering all of the data.
- **No likelihood and no encoder.** A GAN doesn't give the $p(x)$ of an image, and there is no direct way to find
  the $z$ of a real image.

Diffusion models (Lesson 21) have replaced GANs for most image-generation tasks: training is just a stable
regression problem, with a loss that goes down, and they cover the data well. But the adversarial loss lives on:
Stable Diffusion's autoencoder is trained with a patch-wise discriminator so that its compressed images don't come
out blurry (Rombach et al., 2022), and ADD (SDXL Turbo) distills a diffusion model with an adversarial loss to
generate images in 1–4 steps (Sauer et al., 2023).

## Summary
- A GAN has two networks: $D$ is a binary classifier (cross-entropy, Lesson 04), and $G$ learns through gradients
  that pass through $D$. The original loss $\log(1 - D(G(z)))$ saturates when $D$ wins easily: on MNIST, at step
  500, its gradient is about 14,000 times smaller than that of the non-saturating loss $-\log D(G(z))$.
- For a fixed $G$, $D^* = p_{\text{data}} / (p_{\text{data}} + p_g)$ and the game becomes
  $-\log 4 + 2\cdot\mathrm{JSD}$. With $D^*$, the original loss is the gradient of the JSD; the non-saturating loss
  is the gradient of $\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$ (both checked to $10^{-6}$).
- On two Gaussian clusters, the original loss covers both with a wide Gaussian $\mathcal N(0,\, 2.04^2)$ (the JSD
  minimum); the non-saturating loss picks one cluster.
- On the ring of 8 clusters, without R1 the generator covers only a few clusters (mode collapse); with R1 it covers
  all 8 and $D \approx \tfrac12$. A "most populated cluster" that keeps changing is not necessarily mode hopping:
  check whether any good samples are left.
- On MNIST: 4/12 runs fail without R1, 0/18 with it. The best GAN has an FD of 7.88–8.02, on par with Lesson 19's
  VAE + GMM.
- $D \approx \tfrac12$ doesn't mean success: a fresh classifier tells GAN images from real ones 88–89% of the time
  (two sets of real images: 53%). FD is biased by the number of samples; low recall is mode collapse; with 100
  training images, about half of the samples are copies.

## Review questions
1. Why does the original loss $\log(1 - D(G(z)))$ give $G$ almost no gradient early in training, while
   $-\log D(G(z))$ doesn't?
2. With the optimal $D^*$, at which points is $D^*(x) = \tfrac12$? Why isn't "$D$ equals ½ on average" enough to
   conclude that $p_g = p_{\text{data}}$?
3. On two Gaussian clusters, why does the non-saturating loss prefer one cluster to a wide Gaussian covering both,
   even though the wide Gaussian has a smaller JSD?
4. What does R1 penalize, and where? Why does it help even the original loss, which fails 3/3 times on MNIST without
   it?
5. The generator never receives a real image. Why does it still copy training images when there are only 100 of
   them?
6. A GAN has an FD of 8, and a fresh classifier tells its samples from real images 88% of the time. A VAE + GMM has
   an FD of 8 and a C2ST of 71–72%. Which model is better?

<details><summary>Hints and answers</summary>

1. With respect to the logit $a$ of a fake sample, the derivative of $\log(1 - D)$ is $-D$: when $D$ rejects easily
   ($D \to 0$), it dies out. The derivative of $-\log D$ is $-(1 - D)$, close to 1 at exactly that moment. The same
   direction, the same fixed point, different magnitudes.
2. Only where $p_g(x) = p_{\text{data}}(x)$: in section 4.1, at two points. A run's $D$ can give ½ on average
   because it isn't optimal (one step behind, kept smooth by R1); a fresh classifier still tells the two apart 88% of
   the time.
3. With $D^*$, it follows $\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$. The KL in this direction
   heavily penalizes putting probability where there is no data (the gap between the two clusters), and barely
   penalizes missing a cluster.
4. The squared slope $\lVert\nabla_x a(x)\rVert^2$ of $D$, only on real images. It keeps $D$ smooth, so $D$ can't
   reject the fakes with absolute confidence right from the start; $D(G(z))$ doesn't fall to $10^{-5}$, and the
   gradient of the original loss doesn't die out.
5. $D$ learns on real images and can memorize 100 of them; its gradient pulls $G$'s samples toward exactly those
   images. Information about the data still reaches $G$, only indirectly.
6. You can't tell yet: the two metrics look at two different things (LeNet features and pixels). You also have to
   look at precision, recall and the nearest training images, then choose according to what the model will be used
   for.
</details>

## Further reading
- T. Karras, S. Laine & T. Aila (2019), [StyleGAN](https://arxiv.org/abs/1812.04948): FID on 50,000 images.
- I. Goodfellow et al. (2014), [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661), and
  [NIPS 2016 Tutorial: Generative Adversarial Networks](https://arxiv.org/abs/1701.00160).
- M. Arjovsky & L. Bottou (2017), [Towards Principled Methods for Training GANs](https://arxiv.org/abs/1701.04862):
  Theorem 2.5 on the non-saturating loss.
- A. Radford, L. Metz & S. Chintala (2015), [DCGAN](https://arxiv.org/abs/1511.06434): Adam with $\beta_1 = 0.5$.
- L. Metz et al. (2016), [Unrolled GANs](https://arxiv.org/abs/1611.02163): the ring of 8 clusters and mode collapse.
- A. Srivastava et al. (2017), [VEEGAN](https://arxiv.org/abs/1705.07761): counting clusters and good samples on 2-D
  data.
- L. Mescheder, A. Geiger & S. Nowozin (2018),
  [Which Training Methods for GANs do actually Converge?](https://arxiv.org/abs/1801.04406): the R1 gradient penalty.
- F. Farnia & A. Ozdaglar (2020), [GANs May Have No Nash Equilibria](https://arxiv.org/abs/2002.09124).
- Y. Huang et al. (2024), [The GAN is dead; long live the GAN! (R3GAN)](https://arxiv.org/abs/2501.05441): a
  relativistic loss + R1 + R2 with a modern architecture, outperforming StyleGAN2.
- M. Heusel et al. (2017), [FID](https://arxiv.org/abs/1706.08500); M. Bińkowski et al. (2018),
  [Demystifying MMD GANs](https://arxiv.org/abs/1801.01401): FID is biased by the number of samples and can only be
  compared at the same number of samples.
- T. Kynkäänniemi et al. (2019), [Improved Precision and Recall](https://arxiv.org/abs/1904.06991).
- D. Lopez-Paz & M. Oquab (2017), [Revisiting Classifier Two-Sample Tests](https://arxiv.org/abs/1610.06545).
- N. Carlini et al. (2023), [Extracting Training Data from Diffusion Models](https://arxiv.org/abs/2301.13188):
  section 6 compares with GANs.
- M. Kang et al. (2023), [GigaGAN](https://arxiv.org/abs/2303.05511): 512 px images in 0.13 seconds.
- P. Esser, R. Rombach & B. Ommer (2021), [VQGAN](https://arxiv.org/abs/2012.09841); A. Sauer et al. (2023),
  [Adversarial Diffusion Distillation](https://arxiv.org/abs/2311.17042) (SDXL Turbo).
- [GAN Lab](https://poloclub.github.io/ganlab/) (Kahng et al., 2018): a 2-D GAN in the browser.

**Next lesson:** [Lesson 21 · Diffusion & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/diffusion/):
replace the game with a stable regression problem, learning to remove noise step by step.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")